<a href="https://colab.research.google.com/github/novakai-one/claude-ai-demo/blob/main/notebooks/dsa/05_sorting_searching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*DSA topic 5 · Sorting and binary search · practice notebook*

# How do you put things in order fast, and then find one fast?

> **In short**
>
> **How do you put things in order fast, and then find one fast?**
>
> Merge sort splits the list, sorts the halves and merges them: about n log n steps. Once sorted, binary search finds any item by halving the range each time: about log n checks.

## Where you'll end up

In this notebook you will:

1. **Write merge sort** and count its comparisons, then compare it with comparing every pair.
2. **Write binary search**, and meet Python's own versions (`sorted` and `bisect`).
3. **Solve 5 practice problems.** Each has checks you can run, and a solution one click away.

Running every cell takes a few seconds. The site's step-by-step animation of these algorithms is on the
*Sorting and binary search* page of the DSA track.

## How to use this notebook

- Run each grey code cell with **Shift + Enter**, top to bottom.
- 🤔 **Predict first** boxes ask you to guess before you run. The answer is one click away.
- 🐍 **Python notes** explain Python as it comes up.
- Solutions are in collapsed cells. In Colab, double-click a solution's title to open it. Try first; open it when you're stuck.

## How the checks work

Each problem gives you a function to fill in, and a list of test cases.
Run the check after writing your code: it prints ✓ or ✗ for each case.
Run the next cell once now, so `check` exists.

In [1]:
def check(fn, cases):
    """Run fn on each (arguments, expected answer) pair and print ✓ or ✗."""
    passed = 0
    for args, expected in cases:
        try:
            got = fn(*args)
        except Exception as e:                      # a crash counts as a wrong answer
            got = f"error: {type(e).__name__}: {e}"
        ok = got == expected
        passed += ok
        shown = ", ".join(repr(a) for a in args)
        print(("✓" if ok else "✗"), f"{fn.__name__}({shown}) -> {got!r}", "" if ok else f"   (expected {expected!r})")
    print(f"{passed} of {len(cases)} passed")

🐍 **Python notes**
- `def check(fn, cases):` takes a **function** as an argument. In Python, functions are values like any other.
- `fn(*args)` unpacks a tuple into separate arguments: `fn(*(3, 4))` is `fn(3, 4)`.
- `try: ... except Exception as e:` catches an error, so one crash doesn't stop the whole check.
- `f"{got!r}"` is an f-string; `!r` shows the value as Python would write it (strings get quotes).

## 1. Two sorted lists into one

You have two lists that are already sorted. **How do you combine them into one sorted list, quickly?**

Look at the front item of each list. The smaller one must be the smallest of all, so it goes first.
Remove it, and repeat.

### 🤔 Predict first

Merging `[2, 5, 9]` and `[1, 6, 7]` this way: **how many comparisons?**

<details><summary><b>Show me the answer</b> (or skip it, and run the next cell)</summary>

**5.** 1 vs 2, 2 vs 6, 5 vs 6, 9 vs 6, 9 vs 7. Then the second list is empty, so 9 is copied without comparing.

</details>

In [2]:
def merge(a, b, counter):
    out = []
    i = j = 0
    while i < len(a) and j < len(b):
        counter[0] += 1                      # one comparison
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])                        # whatever is left in one list
    out.extend(b[j:])
    return out

counter = [0]
print(merge([2, 5, 9], [1, 6, 7], counter), "comparisons:", counter[0])

[1, 2, 5, 6, 7, 9] comparisons: 5


🐍 **Python notes**
- `counter` is a one-item list, `[0]`. The function changes `counter[0]`, and the caller sees the change, because both refer to the same list.
- `a[i:]` is a **slice**: the items from position `i` to the end. `out.extend(...)` adds all of them.
- `i = j = 0` sets both names to 0.

## 2. Merge sort: split, sort each half, merge

A list of one item is already sorted. Any longer list: split it in half, sort each half the same way, and merge the two sorted halves.

In [3]:
def merge_sort(xs, counter):
    if len(xs) <= 1:
        return xs                            # one item: already sorted
    mid = len(xs) // 2
    left = merge_sort(xs[:mid], counter)
    right = merge_sort(xs[mid:], counter)
    return merge(left, right, counter)

import random
random.seed(0)
xs = random.sample(range(100), 12)
counter = [0]
print(xs)
print(merge_sort(xs, counter), "comparisons:", counter[0])

[49, 97, 53, 5, 33, 65, 62, 51, 38, 61, 45, 74]
[5, 33, 38, 45, 49, 51, 53, 61, 62, 65, 74, 97] comparisons: 31


🐍 `merge_sort` calls itself on smaller lists: **recursion**. The `if len(xs) <= 1` line is the **base case** that stops it.
`len(xs) // 2` is whole-number division: `7 // 2` is 3.

### How many comparisons, compared with checking every pair?

Count merge sort's comparisons for growing lists, next to $n(n-1)/2$ (the number of pairs) and $n \log_2 n$.

In [4]:
import math
print(f"{'n':>7} {'merge sort':>11} {'n log2 n':>10} {'every pair':>12}")
for n in [10, 100, 1_000, 10_000, 100_000]:
    counter = [0]
    merge_sort(random.sample(range(10 * n), n), counter)
    print(f"{n:>7,} {counter[0]:>11,} {round(n * math.log2(n)):>10,} {n * (n - 1) // 2:>12,}")

      n  merge sort   n log2 n   every pair
     10          23         33           45
    100         549        664        4,950
  1,000       8,741      9,966      499,500
 10,000     120,422    132,877   49,995,000


100,000   1,536,222  1,660,964 4,999,950,000


**What you see:** merge sort's count stays a little under $n \log_2 n$. Comparing every pair grows far faster:
at 100,000 items it is about 5 billion pairs against under 2 million comparisons.

**What it's called:** splitting a problem into smaller copies of itself, solving them and combining the answers is **divide and conquer**.
Merge sort takes $O(n \log n)$ time.

## 3. The library version: `sorted`

Python's `sorted(xs)` returns a new sorted list; `xs.sort()` sorts the list in place.
Both use **Timsort**, a merge sort that also spots runs that are already in order.

In [5]:
import time
data = random.sample(range(10_000_000), 200_000)

t0 = time.time(); mine = merge_sort(data, [0]); t_mine = time.time() - t0
t0 = time.time(); theirs = sorted(data); t_lib = time.time() - t0
print("same answer:", mine == theirs)
print(f"your merge sort: {t_mine:.2f}s    sorted(): {t_lib:.3f}s")

people = [("Ana", 31), ("Ben", 25), ("Chloe", 31), ("Dev", 19)]
print(sorted(people, key=lambda p: p[1]))                  # sort by age
print(sorted(people, key=lambda p: (-p[1], p[0])))        # oldest first, then by name

same answer: True
your merge sort: 0.49s    sorted(): 0.040s
[('Dev', 19), ('Ben', 25), ('Ana', 31), ('Chloe', 31)]
[('Ana', 31), ('Chloe', 31), ('Ben', 25), ('Dev', 19)]


🐍 **Python notes**
- `key=lambda p: p[1]` tells `sorted` what to compare: here the second item of each pair. `lambda` makes a small one-line function.
- A key that returns a tuple, like `(-p[1], p[0])`, compares the first item, then the second to break ties. The minus sign reverses the age order.
- `200_000` is the same as `200000`; the underscores only make it easier to read.

**What you see:** the same answer, but `sorted` is many times faster. It is written in C, and it is the one to use in real code.
Writing your own is how you learn what it does and what it costs.

## 4. Find an item by halving: binary search

In a sorted list, check the middle item. If it is too small, the target can only be in the right half; if too big, only in the left half.
Each check throws away half of what is left.

### 🤔 Predict first

A sorted list holds 1,000,000 numbers. **How many checks can binary search need, at most?**

<details><summary><b>Show me the answer</b> (or skip it, and run the next cell)</summary>

**20.** Each check halves the range, and $2^{20} = 1{,}048{,}576$ is the first power of 2 above a million.

</details>

In [6]:
def binary_search(xs, target):
    lo, hi = 0, len(xs) - 1
    checks = 0
    while lo <= hi:
        mid = (lo + hi) // 2
        checks += 1
        if xs[mid] == target:
            return mid, checks
        if xs[mid] < target:
            lo = mid + 1                     # target can only be to the right
        else:
            hi = mid - 1                     # target can only be to the left
    return -1, checks                        # not found

big = list(range(0, 2_000_000, 2))           # 1,000,000 even numbers, sorted
worst = max(binary_search(big, t)[1] for t in random.sample(range(2_000_000), 2_000))
print("position of 777,778:", binary_search(big, 777_778))
print("most checks over 2,000 random searches:", worst)

position of 777,778: (388889, 20)
most checks over 2,000 random searches: 20


🐍 `lo, hi = 0, len(xs) - 1` sets two names at once. A function can return several values as a tuple: `return mid, checks`.

**What it's called:** this is **binary search**. It takes $O(\log n)$ checks, but only works on sorted data.

**The library version:** the `bisect` module. `bisect_left(xs, x)` returns the first position where `x` could be inserted with the list staying sorted.

In [7]:
from bisect import bisect_left, bisect_right
xs = [3, 8, 8, 8, 15, 21]
print("first position of 8:", bisect_left(xs, 8))
print("position after the last 8:", bisect_right(xs, 8))
print("how many 8s:", bisect_right(xs, 8) - bisect_left(xs, 8))

first position of 8: 1
position after the last 8: 4
how many 8s: 3


## 5. Halving works on more than lists

**What is the smallest whole number $x$ with $x^2 \ge 1{,}000{,}000{,}007$?**

The yes/no question "is $x^2 \ge$ the target?" is *no* for small $x$ and *yes* for every larger one: it flips once.
Halving finds the flip point, without any list at all.

In [8]:
def smallest_yes(lo, hi, is_yes):
    """Smallest x in [lo, hi] with is_yes(x) True. Assumes the answer flips from False to True once."""
    while lo < hi:
        mid = (lo + hi) // 2
        if is_yes(mid):
            hi = mid                         # mid works: the answer is mid or smaller
        else:
            lo = mid + 1                     # mid fails: the answer is bigger
    return lo

target = 1_000_000_007
x = smallest_yes(0, target, lambda v: v * v >= target)
print(x, x * x >= target, (x - 1) * (x - 1) >= target)

31623 True False


**What you see:** about 30 checks find the answer among a billion candidates.

**What it's called:** this is **binary search on the answer**. It works whenever a yes/no question flips once as the number grows.

## Practice

Fill in each function, then run its cell to see the checks. The method isn't named: deciding how to solve it is part of the problem.

### Problem 1

A sorted list may contain repeats. Return the **first position holding a value of at least `x`**.
If every value is smaller than `x`, return `len(xs)`. Aim for about $\log n$ steps.

In [9]:
def first_at_least(xs, x):
    # your code here
    return None

CASES_1 = [(([1, 3, 3, 3, 8], 3), 1), (([1, 3, 3, 3, 8], 4), 4), (([1, 3, 3, 3, 8], 9), 5),
           (([1, 3, 3, 3, 8], 0), 0), (([], 5), 0), (([7], 7), 0)]
check(first_at_least, CASES_1)

✗ first_at_least([1, 3, 3, 3, 8], 3) -> None    (expected 1)
✗ first_at_least([1, 3, 3, 3, 8], 4) -> None    (expected 4)
✗ first_at_least([1, 3, 3, 3, 8], 9) -> None    (expected 5)
✗ first_at_least([1, 3, 3, 3, 8], 0) -> None    (expected 0)
✗ first_at_least([], 5) -> None    (expected 0)
✗ first_at_least([7], 7) -> None    (expected 0)
0 of 6 passed


In [ ]:
#@title Solution to problem 1 (double-click to show)
def first_at_least(xs, x):
    lo, hi = 0, len(xs)                      # the answer is somewhere in 0..len(xs)
    while lo < hi:
        mid = (lo + hi) // 2
        if xs[mid] >= x:
            hi = mid                         # mid could be the answer; look left
        else:
            lo = mid + 1
    return lo

check(first_at_least, CASES_1)
# This is what bisect_left does.

### Problem 2

A sorted list of exam marks. **How many marks are between `lo` and `hi`, including both ends?**
The list can be very long, so don't walk through all of it.

In [11]:
def count_between(marks, lo, hi):
    # your code here
    return None

CASES_2 = [(([40, 52, 52, 67, 71, 88, 93], 50, 70), 3), (([40, 52, 52, 67, 71, 88, 93], 0, 100), 7),
           (([40, 52, 52, 67, 71, 88, 93], 94, 99), 0), (([40, 52, 52, 67, 71, 88, 93], 52, 52), 2)]
check(count_between, CASES_2)

✗ count_between([40, 52, 52, 67, 71, 88, 93], 50, 70) -> None    (expected 3)
✗ count_between([40, 52, 52, 67, 71, 88, 93], 0, 100) -> None    (expected 7)
✗ count_between([40, 52, 52, 67, 71, 88, 93], 94, 99) -> None    (expected 0)
✗ count_between([40, 52, 52, 67, 71, 88, 93], 52, 52) -> None    (expected 2)
0 of 4 passed


In [ ]:
#@title Solution to problem 2 (double-click to show)
from bisect import bisect_left, bisect_right

def count_between(marks, lo, hi):
    # first position >= lo, and first position > hi: everything between them is in range
    return bisect_right(marks, hi) - bisect_left(marks, lo)

check(count_between, CASES_2)

### Problem 3

A list of meetings, each a pair `(start, end)`, in no order. Meetings that overlap or touch must share one room booking.
**Return the combined bookings, earliest first.** For example, `(1, 3)` and `(2, 6)` become `(1, 6)`.

In [13]:
def combine(meetings):
    # your code here
    return None

CASES_3 = [(([(1, 3), (2, 6), (8, 10), (15, 18)],), [(1, 6), (8, 10), (15, 18)]),
           (([(5, 7), (1, 2), (2, 4)],), [(1, 4), (5, 7)]),
           (([(1, 10), (2, 3), (4, 5)],), [(1, 10)]),
           (([],), [])]
check(combine, CASES_3)

✗ combine([(1, 3), (2, 6), (8, 10), (15, 18)]) -> None    (expected [(1, 6), (8, 10), (15, 18)])
✗ combine([(5, 7), (1, 2), (2, 4)]) -> None    (expected [(1, 4), (5, 7)])
✗ combine([(1, 10), (2, 3), (4, 5)]) -> None    (expected [(1, 10)])
✗ combine([]) -> None    (expected [])
0 of 4 passed


In [ ]:
#@title Solution to problem 3 (double-click to show)
def combine(meetings):
    out = []
    for start, end in sorted(meetings):              # sorted by start time
        if out and start <= out[-1][1]:              # overlaps or touches the last booking
            out[-1] = (out[-1][0], max(out[-1][1], end))
        else:
            out.append((start, end))
    return out

check(combine, CASES_3)
# Sorting first means each meeting only needs comparing with the last booking: O(n log n) in total.

### Problem 4

A printer has a list of jobs, given as page counts. It works on one job at a time.
Each hour it prints up to `s` pages of the current job; if the job finishes early, the rest of that hour is unused.
**What is the smallest whole number `s` that finishes every job within `h` hours?** (Assume `h` is at least the number of jobs.)

In [15]:
def slowest_speed(jobs, h):
    # your code here
    return None

CASES_4 = [(([3, 6, 7, 11], 8), 4), (([30, 11, 23, 4, 20], 5), 30), (([30, 11, 23, 4, 20], 6), 23),
           (([1], 1), 1), (([1_000_000_000], 2), 500_000_000)]
check(slowest_speed, CASES_4)

✗ slowest_speed([3, 6, 7, 11], 8) -> None    (expected 4)
✗ slowest_speed([30, 11, 23, 4, 20], 5) -> None    (expected 30)
✗ slowest_speed([30, 11, 23, 4, 20], 6) -> None    (expected 23)
✗ slowest_speed([1], 1) -> None    (expected 1)
✗ slowest_speed([1000000000], 2) -> None    (expected 500000000)
0 of 5 passed


In [ ]:
#@title Solution to problem 4 (double-click to show)
def hours_needed(jobs, s):
    return sum((pages + s - 1) // s for pages in jobs)    # whole hours per job, rounded up

def slowest_speed(jobs, h):
    lo, hi = 1, max(jobs)                    # speed max(jobs) always works: one hour per job
    while lo < hi:
        mid = (lo + hi) // 2
        if hours_needed(jobs, mid) <= h:
            hi = mid                         # fast enough: try slower
        else:
            lo = mid + 1                     # too slow
    return lo

check(slowest_speed, CASES_4)
# "Fast enough?" is False for small s and True for every larger s, so halving finds the flip point.

### Problem 5

A list of numbers in no order. **What is the smallest difference between any two of them?**
Checking every pair is too slow for a million numbers.

In [17]:
def smallest_gap(xs):
    # your code here
    return None

CASES_5 = [(([31, 4, 15, 9, 26],), 5), (([10, 3, 10],), 0), (([1, 100],), 99), (([-5, 7, 2, -1],), 3)]
check(smallest_gap, CASES_5)

✗ smallest_gap([31, 4, 15, 9, 26]) -> None    (expected 5)
✗ smallest_gap([10, 3, 10]) -> None    (expected 0)
✗ smallest_gap([1, 100]) -> None    (expected 99)
✗ smallest_gap([-5, 7, 2, -1]) -> None    (expected 3)
0 of 4 passed


In [ ]:
#@title Solution to problem 5 (double-click to show)
def smallest_gap(xs):
    s = sorted(xs)
    return min(b - a for a, b in zip(s, s[1:]))     # after sorting, the closest pair are neighbours

check(smallest_gap, CASES_5)
# zip(s, s[1:]) pairs each item with the next one. Sorting costs O(n log n); the scan costs O(n).

## Recognition cues

- When you see **find an item in sorted data** in a problem, think **binary search (or `bisect`)**.
- When you see **"the smallest x such that ..." with a yes/no answer that flips once** in a problem, think **binary search on the answer**.
- When you see **closest pair, overlaps, duplicates** in a problem, think **sort first, then look at neighbours**.
- When you see **sorting up to $10^6$ items** in a problem, think **an $O(n \log n)$ sort, which `sorted` already is**.

## Practise more

For this topic: the CSES Problem Set's *Sorting and Searching* section; the USACO Guide's *Binary Search* module (Silver);
LeetCode's *Binary Search* and *Sorting* tags.

- CSES Problem Set: https://cses.fi/problemset/ (short problems grouped by topic, with an online judge)
- USACO Guide: https://usaco.guide/ (free modules from Bronze to Platinum)
- LeetCode: https://leetcode.com/problemset/ (filter by topic tag)

Pick two or three problems, solve them here in a new cell first, then submit.